# 5-LOX model training

Run the cells top to bottom. Only **cell 1** needs editing.

**What changed:** the raw table now carries the descriptor columns already, so
nothing is featurised during training. Features are read straight from the
table. SMILES are only needed for the similarity checks and for scoring new
molecules; the notebook detects whether they are present and skips what it cannot do.

**Runtime:** roughly 30-50 minutes with the default settings. The GPU is not
used; scikit-learn random forests run on CPU. More cores means faster.

**Needs:** `rdkit scikit-learn optuna shap joblib pandas numpy matplotlib scipy`

## 1. Settings — edit this cell

In [ ]:
# Column names in the raw CSV. None = detect automatically.
LABEL_COL = None   # e.g. "Activity"
SMILES_COL = None  # e.g. "Smiles"; fine to leave as None if there is none

# Drop rows whose feature vectors are identical. The descriptors are
# stereo-blind, so stereoisomers become duplicate rows; where they disagree
# about the label the model cannot resolve them and they leak across splits.
DEDUPE = True

# How long the hyperparameter search may run, PER split mode.
# Stops at whichever limit is reached first. "both" runs it twice.
N_TRIALS = 30
TIMEOUT_MIN = 45

# "both" = random + scaffold (recommended: the gap is a result in itself)
SPLIT_MODE = "both"

# Explainability
TARGET_CLASS = 1          # class to explain, 1 = active
ONLY_IN_DOMAIN = True    # explain only test molecules similar to the training data
SHAP_MAX_MOLECULES = 500  # TreeSHAP on a large forest is slow, so subsample

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import ethnohack.cox_model.lox5_preprocess as prep
import ethnohack.cox_model.lox5_train as train

## 2. Load the raw data and check the columns

In [ ]:
raw = prep.load_raw()
print(raw.columns.tolist())
raw.head(3)

## 3. Preprocessing

Renames the label to `label` (so it cannot be mistaken for a feature), removes
duplicate and contradictory feature vectors, and keeps the existing scaffold
column. No features are computed.

In [ ]:
data = prep.build_dataset(
    label_col=LABEL_COL,
    smiles_col=SMILES_COL,
    dedupe=DEDUPE,
)

HAS_SMILES = "smiles" in data.columns
cols = prep.feature_columns(data)

print(f"\nSMILES available: {HAS_SMILES}")
print(f"{len(cols)} feature columns")
print(f"{data['scaffold'].nunique()} scaffolds for {len(data)} compounds")

### Checks

`leaked metadata` must be empty. A label in the feature list makes the model
score near 1.0 and mean nothing.

In [ ]:
print("leaked metadata:",
      [c for c in ("ChEMBL_IDs", "Activity", "label", "scaffold") if c in cols]
      or "none")
print("non-numeric features:",
      [c for c in cols if not pd.api.types.is_numeric_dtype(data[c])] or "none")
print("constant columns:", int((data[cols].nunique() <= 1).sum()))
print("NaNs:", int(data[cols].isna().sum().sum()))
print("label values:", sorted(data["label"].unique()),
      f"| active share {data['label'].mean():.1%}")

### Do training and prediction compute the same features?

This is the most important check in the pipeline. Training reads descriptor
columns from a file; prediction recomputes them from a SMILES. If the two
disagree, the model silently receives wrong input on the plant compounds.

Without a SMILES column this cannot be verified, and you are trusting that the
descriptors came from plain RDKit defaults with no scaling applied afterwards.

In [ ]:
prep.verify_features(data)

## 4. Train

**Random split:** close analogues land in both train and test, so the model can
recognise the compound series instead of the chemistry. Scores look good.

**Scaffold split:** molecules sharing a ring core stay on one side. Closer to
the real task (unseen plant metabolites). Lower, but honest. **Report this one.**

This is the long cell.

In [ ]:
results = train.run_models(
    data, SPLIT_MODE, n_trials=N_TRIALS, timeout_min=TIMEOUT_MIN
)
best = results[-1]  # scaffold run when SPLIT_MODE == "both"

In [ ]:
train.plot_roc(results)

## 5. Sanity checks

These decide what you can claim about the model. Note the numbers down.
Both need SMILES, so they are skipped automatically without them.

### 5a. Is the model better than a similarity search?

Baseline: score each test compound by how similar it is to the nearest known
active. If the model barely beats this, it mostly learned "looks like a known
active" rather than anything about the chemistry. Worth reporting either way.

In [ ]:
if HAS_SMILES:
    sim = train.similarity_baseline(best)
    combo = train.combined_baseline(best, sim)
else:
    print("Skipped: needs a SMILES column. Note this on the limitations slide.")

### 5b. Where does the model actually work?

Test AUC split by structural similarity to the training set. The lowest bin
predicts how the model will behave on unfamiliar plant metabolites, which is
the number that matters for the project.

In [ ]:
if HAS_SMILES:
    bins = train.reliability_bins(best)
else:
    print("Skipped: needs a SMILES column.")

## 6. Save

Saves the model together with the exact feature column order, plus the training
SMILES when available (needed for the reliability flag at prediction time).

`tune_cutoff=True` picks the 0/1 threshold by cross-validation on the training
data instead of defaulting to 0.5. The test set stays untouched.

In [ ]:
train.save_model(best, tune_cutoff=True)

## 7. Check the saved model loads

`featurize` recomputes the descriptors from SMILES, so this also proves the
prediction path works end to end.

In [ ]:
import ethnohack.cox_model.lox5_model_api as lox

lox.reset_cache()
lox.model_info()

lox.predict(["CC(=O)Oc1ccccc1C(=O)O", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"])  # aspirin, ibuprofen

---

## Reading the results

**Compare PR-AUC against the prevalence**, not against 0. Prevalence is the
share of actives, which is what random guessing already achieves.

**The probabilities are a ranking, not a verdict.** Use them to compare and
sort compounds, not to declare one active.

**Filter to `in_domain == True`** before drawing conclusions or running SHAP,
where that column is available.

## For the 5-LOX model

Point the constants at the 5-LOX paths and rerun this notebook unchanged.

---

## 8. Explain the model (TreeSHAP)

SHAP splits each prediction into contributions of the individual features
(fingerprint bits and any extra descriptors). Positive values push toward
active, negative toward inactive.

Uses the saved model from section 7, so the features match the training exactly.

In [ ]:
import ethnohack.explainer as xai

model = lox.load_model()

# fingerprint bits must come first, extra features after (needed for bit drawing)
xai.check_feature_layout(lox.feature_names(), model, lox.load_bundle())

### 8a. Global view: which features drive the model

With `ONLY_IN_DOMAIN = True` only test molecules similar enough to the training
data are used, because SHAP on unfamiliar molecules explains unreliable predictions.

In [ ]:
test_smiles = best["test_df"]["smiles"]
in_domain = lox.predict(test_smiles)["in_domain"].to_numpy(dtype=bool)
use = in_domain | (not ONLY_IN_DOMAIN)
print(f"{use.sum()} of {len(use)} test molecules used")

X_te = lox.featurize(test_smiles)[use]

# fingerprint bits get a substructure hint in their name for the plots
feature_names = xai.readable_feature_names(lox.feature_names(), test_smiles[use])

shap_matrix, X_shap = xai.global_shap(
    model, X_te, TARGET_CLASS, n_max=SHAP_MAX_MOLECULES
)
xai.plot_global_shap(shap_matrix, X_shap, feature_names)

In [ ]:
top_features = xai.top_bits_table(shap_matrix, X_shap, feature_names)
top_features

In [ ]:
# most important fingerprint bit drawn as a substructure on a test molecule
top_bits = xai.top_bits_table(shap_matrix, X_shap, feature_names, n_top=1, fp_only=True)
xai.highlight_bit(top_bits["feature_index"].iloc[0], X_te, test_smiles[use])

### 8b. Local view: why single molecules get their prediction

Most confident hits and mistakes from the test set. Red = pushes toward active,
blue = pushes toward inactive. The waterfall starts at the average prediction
(base value) and ends at the model probability.

In [ ]:
examples = xai.select_examples(best, mask=use)

# in_domain tells you whether each prediction can be trusted
lox.predict([smiles for smiles, _, _ in examples.values()])

In [ ]:
for name, (smiles, proba, label) in examples.items():
    print(f"\n##### {name} | True Label: {label} | Model Probability: {proba:.3f}")
    xai.explain_molecule(
        smiles,
        model,
        target_class=TARGET_CLASS,
        feature_names=feature_names,
        featurizer=lox.featurize,
    )

## 9. Screen COCONUT natural products

Scores natural products from COCONUT with the saved model, checks whether each
one is similar enough to the training data to be trusted, ranks them and
explains the top hits.

**Before you start:** download the COCONUT CSV from
https://coconut.naturalproducts.net/download and put it at `COCONUT_CSV` in
`constants.py`. Sections 6 and 7 must have run, because this uses the saved model.

In [ ]:
import importlib
import ethnohack.constants
import ethnohack.coconut as coco
importlib.reload(ethnohack.constants)
from ethnohack.constants import PLANT_GENERA_FILE

In [ ]:
# COCONUT screening
COCONUT_SMILES_COL = "canonical_smiles"
COCONUT_ID_COL = "identifier"
COCONUT_NAME_COL = "name"
COCONUT_N_MOLECULES = None  # random sample to test the workflow, None = all molecules
COCONUT_CHUNK_SIZE = 5000  # molecules scored per batch, lower = less memory
COCONUT_TOP_N = 50  # how many top-ranked molecules to list
COCONUT_N_EXPLAIN = 10  # how many of those to explain with SHAP
COCONUT_ORGANISM_COL = "organisms"   # check with coco.list_columns()

In [ ]:
coco.list_columns()   # first run only: confirm the organism column name

genera = coco.load_plant_genera(PLANT_GENERA_FILE)
coconut, np_overlap = coco.prepare_coconut(
    plant_genera=genera,
    organism_col=COCONUT_ORGANISM_COL,
    smiles_col=COCONUT_SMILES_COL,
    id_col=COCONUT_ID_COL,
    name_col=COCONUT_NAME_COL,
    n_max=COCONUT_N_MOLECULES,
)
coconut.head()

### 9a. Score

Includes the reliability check (`max_sim_train`, `in_domain`). This is the slow
part, because every molecule is compared with the whole training set. Start with
the sample from cell 1, then set `COCONUT_N_MOLECULES = None` for the full run.

In [ ]:
scored = coco.score_coconut(coconut, chunk_size=COCONUT_CHUNK_SIZE)
coco.summarize(scored)

In [ ]:
cand = scored[scored["max_sim_train"] >= 0.49].reset_index(drop=True)
cand = coco.flag_alerts(cand)

### 9b. Top hits

Only `in_domain` molecules are ranked, because predictions on molecules far
from the training data are much less reliable. The probability is a ranking,
not a verdict: use it to compare and prioritise, not to call a compound active.

In [ ]:
hits = coco.top_hits(
    cand,
    n=COCONUT_TOP_N,
    in_domain_only=False,
    min_confidence="medium",
    exclude_alerts=True,
)
hits

In [ ]:
coco.save_scores(scored)

### 9c. Explain the top hits

Red = pushes toward active, blue = pushes toward inactive. The waterfall starts
at the average prediction (base value) and ends at the model probability.

In [ ]:
import ethnohack.explainer as xai

model = lox.load_model()
top = hits.head(COCONUT_N_EXPLAIN)

# fingerprint bits get a substructure hint in their name for the plots
coconut_names = xai.readable_feature_names(lox.feature_names(), top["std_smiles"])

for _, row in top.iterrows():
    print(f"\n##### {row['id']} | {row['name']} | model probability: {row['proba']:.3f}")
    xai.explain_molecule(
        row["std_smiles"],
        model,
        target_class=TARGET_CLASS,
        feature_names=coconut_names,
        featurizer=lox.featurize,
    )